# Exploring the Dataset: inet-firewall dnsmasq.log (DNS Events)

**Goal:** Understand the structure of `inet-firewall/logs/dnsmasq.log` (dnsmasq syslog) to design the `dns_events` table.

This notebook walks through:
1. Loading the raw dnsmasq log (275,900 lines) and the ground truth labels (54,035 labeled lines)
2. Parsing the syslog format (timestamp, host, process, message) and message action types (query, forwarded, reply, cached, nameserver, failed)
3. Exploring fields and event distribution
4. Building a raw 1:1 DataFrame (message as single TEXT; parsed columns for analysis only)
5. Integrating ground truth labels (DNS exfiltration / DNSteal)
6. Mapping to planned SQL schema (PostgreSQL and MySQL) and checking 1NF, 2NF, 3NF

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** inet-firewall (edge firewall; dnsmasq provides DNS)  
**Findings doc:** `docs/data_exploration/notebook_findings/hunt_dns_logs_findings.md`  
**Scope:** `docs/data_exploration/data_scope_and_findings.md`  
**Normalization rules:** `docs/data_exploration/normalization_rules_sheet.md`

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder. Both the gather (raw log) and labels (JSONL) paths refer to the same logical log; ETL joins on line number.

**Default:** `russellmitchell/` at the same level as the repo.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("..") / ".." / "russellmitchell"
DNS_LOG = DATASET_ROOT / "gather" / "inet-firewall" / "logs" / "dnsmasq.log"
LABEL_FILE = DATASET_ROOT / "labels" / "inet-firewall" / "logs" / "dnsmasq.log"

for p, name in [
    (DATASET_ROOT, "Dataset root"),
    (DNS_LOG, "DNS log"),
    (LABEL_FILE, "Label file"),
]:
    status = "FOUND" if p.exists() else "MISSING"
    print(f"{name}: {p.resolve()} [{status}]")

## 1. Load Raw Data

The dnsmasq log uses syslog-style lines:
```
Mon DD HH:MM:SS host process[pid]: message
```

The **message** (after `dnsmasq[pid]: `) has several forms:
- `query[A] <domain> from <client_ip>` (or query[TXT], query[AAAA])
- `forwarded <domain> to <upstream_ip>`
- `reply <domain> is <result>` / `cached <domain> is <result>`
- `nameserver <free text>`
- `failed` (single occurrence)

Raw timestamp has no year; we infer 2022 from scenario (data_scope_and_findings.md).

In [ ]:
with open(DNS_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

## 2. Parse the dnsmasq Log Format

### 2.1 Event action inventory

Extract the first token of each message (event action) and count.

In [ ]:
import re
from collections import Counter

# Syslog: "Mon DD HH:MM:SS host process[pid]: message"
action_counts = Counter()
for line in raw_lines:
    m = re.search(r"dnsmasq\[\d+\]:\s+(\S+)", line)
    if m:
        token = m.group(1)
        # Normalize query[A] / query[TXT] to action "query" for counting
        if token.startswith("query"):
            action_counts["query"] += 1
        else:
            action_counts[token] += 1

print(f"Distinct message actions: {len(action_counts)}")
print()
for action, count in action_counts.most_common():
    print(f"  {action:15s} {count:6d} ({count / len(raw_lines) * 100:5.1f}%)")

### 2.2 Parsing strategy

Parse each line into: `line_number`, `event_timestamp`, `host`, `process`, `message` (full blob for raw table), and for analysis only: `event_action`, `query_type`, `domain`, `client_ip`, `upstream_ip`, `reply_value`. The raw table keeps `message` as a single TEXT column (1NF violation when stored as one cell); parsed columns are for exploration and normalization design.

In [ ]:
from datetime import UTC, datetime

import pandas as pd

MONTH = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
INFERRED_YEAR = 2022  # From data_scope_and_findings.md (Jan 21-25, 2022)


def parse_dnsmasq_line(line_num, line):
    """Parse one dnsmasq syslog line. Returns dict with line_number, event_timestamp, host, process, message, and parsed sub-fields."""
    record = {"line_number": line_num}
    line = line.rstrip()

    # Syslog: "Mon DD HH:MM:SS host process[pid]: message"
    m = re.match(r"^(\S+)\s+(\d+)\s+(\d{2}:\d{2}:\d{2})\s+(\S+)\s+(\S+)\[(\d+)\]:\s+(.*)$", line)
    if not m:
        record["event_timestamp"] = None
        record["host"] = None
        record["process"] = None
        record["message"] = line
        record["event_action"] = None
        return record

    month_str, day, time_str, host, proc_name, pid, message = m.groups()
    record["host"] = host
    record["process"] = f"{proc_name}[{pid}]"
    record["message"] = message

    try:
        mo, d, t = MONTH.get(month_str, 1), int(day), time_str
        dt = datetime(INFERRED_YEAR, mo, d, int(t[:2]), int(t[3:5]), int(t[6:8]), tzinfo=UTC)
        record["event_timestamp"] = dt
    except (ValueError, KeyError):
        record["event_timestamp"] = None

    # Parse message into event_action and type-specific fields
    record["event_action"] = None
    record["query_type"] = None
    record["domain"] = None
    record["client_ip"] = None
    record["upstream_ip"] = None
    record["reply_value"] = None

    if message.startswith("query"):
        record["event_action"] = "query"
        qt = re.match(r"query\[(A|TXT|AAAA)\]", message)
        if qt:
            record["query_type"] = qt.group(1)
            rest = message[qt.end():].strip()
        else:
            rest = message[5:].strip()  # "query "
        if " from " in rest:
            domain, _, client_ip = rest.partition(" from ")
            record["domain"] = domain.strip()
            record["client_ip"] = client_ip.strip()
    elif message.startswith("forwarded "):
        record["event_action"] = "forwarded"
        rest = message[10:].strip()
        if " to " in rest:
            domain, _, upstream_ip = rest.partition(" to ")
            record["domain"] = domain.strip()
            record["upstream_ip"] = upstream_ip.strip()
    elif message.startswith("reply "):
        record["event_action"] = "reply"
        rest = message[6:].strip()
        if " is " in rest:
            domain, _, reply_val = rest.partition(" is ")
            record["domain"] = domain.strip()
            record["reply_value"] = reply_val.strip()
    elif message.startswith("cached "):
        record["event_action"] = "cached"
        rest = message[7:].strip()
        if " is " in rest:
            domain, _, reply_val = rest.partition(" is ")
            record["domain"] = domain.strip()
            record["reply_value"] = reply_val.strip()
    elif message.startswith("nameserver "):
        record["event_action"] = "nameserver"
    elif message.strip() == "failed":
        record["event_action"] = "failed"

    return record


parsed = [parse_dnsmasq_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {list(parsed[0].keys())}")
print(f"  event_action={parsed[0]['event_action']}, query_type={parsed[0]['query_type']}, domain len={len(parsed[0]['domain'] or '') or 0}, client_ip={parsed[0]['client_ip']}")

In [ ]:
df = pd.DataFrame(parsed)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
for col in df.columns:
    non_null = df[col].notna().sum()
    nunique = df[col].nunique()
    print(f"  {col:20s}  non-null: {non_null:6d}/{len(df)}  unique: {nunique}")

## 3. Field-by-Field Exploration

### 3.1 Event action distribution

In [ ]:
print("=== Event action distribution ===")
action_dist = df["event_action"].value_counts()
for action, count in action_dist.items():
    print(f"  {action:15s} {count:6d} ({count / len(df) * 100:5.1f}%)")

### 3.2 Timestamp range and query_type (query lines only)

In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['event_timestamp'].min()}")
print(f"  Latest:   {df['event_timestamp'].max()}")
print()

query_df = df[df["event_action"] == "query"]
print("=== Query type (query lines only) ===")
print(query_df["query_type"].value_counts(dropna=False).to_string())

In [ ]:
print("=== client_ip (query lines) ===")
print(query_df["client_ip"].value_counts().head(10).to_string())
print()
print("=== upstream_ip (forwarded lines) ===")
fwd_df = df[df["event_action"] == "forwarded"]
print(fwd_df["upstream_ip"].value_counts().to_string())

### 3.3 Domain length and exfiltration pattern

DNSteal encodes data in subdomains; labeled exfiltration domains (e.g. `*.kennedy-mendoza.info`) are very long.

In [ ]:
with_domain = df[df["domain"].notna() & (df["domain"] != "")]
with_domain = with_domain.copy()
with_domain["domain_len"] = with_domain["domain"].str.len()
print("Domain length (where present):")
print(f"  min={with_domain['domain_len'].min()}, max={with_domain['domain_len'].max()}, median={with_domain['domain_len'].median():.0f}")
print()
kennedy = with_domain[with_domain["domain"].str.contains("kennedy-mendoza.info", na=False)]
print(f"Domains containing 'kennedy-mendoza.info': {len(kennedy)} (exfiltration pattern)")

## 4. Raw Table Build

Per **Raw = truly raw** (normalization_rules_sheet and hunt_dns_logs_findings): keep `message` as a single TEXT column. Do **not** store parsed sub-fields (event_action, query_type, domain, client_ip, upstream_ip, reply_value) in the raw DDL; they remain in `df` for analysis only. Labels will be joined by line number and stored as multi-valued columns (dns_event_category, dns_signature_matches).

In [ ]:
raw_cols = ["line_number", "event_timestamp", "host", "process", "message"]
df_raw = df[raw_cols].copy()

print(f"Raw DataFrame shape: {df_raw.shape}")
print(f"Raw columns: {list(df_raw.columns)}")
print()
print("First 2 rows (message truncated):")
pd.set_option("display.max_colwidth", 80)
print(df_raw.head(2).to_string())

## 5. Label Integration

Load the ground truth labels (JSONL). Each line: `{"line": <n>, "labels": [...], "rules": {...}}`. Labels mark DNS exfiltration (DNSteal); all labeled lines have `dnsteal`, `attacker`, `dnsteal-received` and rules `dnsteal.domain.match`, `dnsteal.domain.received`.

In [ ]:
import json

labels = []
with open(LABEL_FILE) as f:
    for line in f:
        labels.append(json.loads(line))

print(f"Labeled lines: {len(labels)}")
print(f"Labeled line numbers (first 10): {[lbl['line'] for lbl in labels[:10]]}")
print()
sample = labels[0]
print(f"Sample: line={sample['line']}, labels={sample['labels']}, rules keys={list(sample['rules'].keys())}")

In [ ]:
from collections import Counter

label_counter = Counter()
for lbl in labels:
    for name in lbl["labels"]:
        label_counter[name] += 1
print("=== Label distribution ===")
for name, count in label_counter.most_common():
    print(f"  {name}: {count} lines")

rule_counter = Counter()
for lbl in labels:
    for rule_list in lbl["rules"].values():
        for r in rule_list:
            rule_counter[r] += 1
print()
print("=== Rule distribution ===")
for r, c in rule_counter.most_common():
    print(f"  {r}: {c}")

In [ ]:
labeled_lines = {lbl["line"] for lbl in labels}
df_labeled = df_raw[df_raw["line_number"].isin(labeled_lines)]
print(f"Labeled records matched: {len(df_labeled)}")
print()
print("Sample labeled records (message truncated):")
print(df_labeled.head(3).to_string())

## 6. Summary Statistics

In [ ]:
print("=== Summary ===")
print(f"Total lines:        {len(df_raw)}")
print(f"Distinct actions:  {df['event_action'].nunique()}")
print(f"Time range:        {df['event_timestamp'].min()} to {df['event_timestamp'].max()}")
print(f"Labeled lines:     {len(labels)} ({len(labels)/len(df_raw)*100:.1f}%)")
print(f"Raw table columns: {list(df_raw.columns)}")

## 7. Schema Mapping and DDL

Map to `dns_events_raw` (raw 1:1 load). Labels and rules are stored as multi-valued columns (1NF violation; normalization unpacks to `attack_labels`). Per hunt_dns_logs_findings: message as TEXT blob; no parsed message sub-fields in raw DDL.

In [ ]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE dns_events_raw (
    dns_event_id            SERIAL PRIMARY KEY,
    line_number             INTEGER NOT NULL,
    event_timestamp         TIMESTAMP WITH TIME ZONE,
    host                    VARCHAR(50),
    process                 VARCHAR(100),
    message                 TEXT NOT NULL,
    dns_event_category      TEXT[],
    dns_signature_matches   JSONB,
    created_at              TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE dns_events_raw (
    dns_event_id            INT AUTO_INCREMENT PRIMARY KEY,
    line_number             INT NOT NULL,
    event_timestamp         DATETIME,
    host                    VARCHAR(50),
    process                 VARCHAR(100),
    message                 TEXT NOT NULL,
    dns_event_category      JSON,
    dns_signature_matches   JSON,
    created_at              DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)

## 8. Normalization Observations

Applying `normalization_rules_sheet.md` and hunt_dns_logs_findings.

### 8.1 1NF Check

**Composite field (message):** The message column embeds multiple attributes (action, query type, domain, client_ip, upstream_ip, reply value) in one text blob. **1NF violated.** Resolution: retain raw message; parse into atomic columns during normalization.

**Multi-valued fields (labels file):** `labels` is an array; `rules` is a nested dict. **1NF violated.** Stored as TEXT[]/JSONB (PostgreSQL) and JSON (MySQL) in raw load; normalize to `attack_labels`.

**Repeating groups:** None.

### 8.2 2NF Check

Raw table uses single-column PK (`dns_event_id`). **2NF satisfied.**

### 8.3 3NF Check

**Transitive dependency:** `event_action` (message prefix) determines which parsed sub-fields exist (query → domain, client_ip; forwarded → domain, upstream_ip; reply/cached → domain, reply_value). So dns_event_id → event_action → populated field set. **3NF violated.** Resolution: event_action/event_type column and type-specific columns or subtype tables in normalized schema.

### 8.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) |
|----|-------------|---------------|
| FD1 | dns_event_id | all attributes |
| FD2 | line_number | all attributes |
| FD3 | event_action | populated field set |
| FD4 | (client_ip, domain, timestamp) | session/flow grouping |

## 9. Key Findings for Schema Design

1. **Single log, two paths:** gather/.../dnsmasq.log and labels/.../dnsmasq.log join on line number to build the same raw → normalized tables.

2. **Message is composite:** Raw table keeps one TEXT column; normalization parses into event_action, query_type, domain, client_ip, upstream_ip, reply_value.

3. **1NF violations:** message (composite); labels and rules (multi-valued). Store labels/rules as TEXT[]/JSONB or JSON; unpack to attack_labels in normalization.

4. **3NF violation:** event_action → field set. Document in normalization report; use event_action and type-specific columns or subtype tables.

5. **Exfiltration subset:** 54,035 labeled lines (~19.6%) are DNSteal; client IP 10.143.0.103, domain pattern *.kennedy-mendoza.info. Unlabeled traffic is normal DNS (ClamAV, etc.) for baseline.

6. **No year in syslog:** Infer year (2022) from scenario when building event_timestamp.

7. **Cross-log correlation:** Exfiltration client 10.143.0.103 can be correlated with other logs by IP and time; attacker VPN IP 172.19.131.174 appears in Apache/auth/audit.